# Train model and log with MLflow

In this project, Week 3 MLflow is about using MLflow to track experiments and manage models for time series forecasting.

1. Creates an MLflow experiment called timeseries_project_forecasting
2. Logs parameters (model type, alpha), metrics (MSE, R²), and serialized models for Linear Regression and Ridge
3. Shows how to load logged models back for inference
4. Demonstrates launching the MLflow UI (mlflow ui) to visually compare runs

The tracked runs are stored in mlflow.db (SQLite) and mlruns/ (artifacts). It pairs with Week_3_Hyperopt.ipynb — Hyperopt tunes XGBoost hyperparameters, MLflow logs and manages the results.

In [ ]:
import mlflow
import mlflow.sklearn
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np
import os

In [ ]:
# Set the tracking URI to your custom folder (optional)
#mlflow.set_tracking_uri(f"file://{os.getcwd()}/mlruns")

experiment_name = "timeseries_project_forecasting"
# Create or set experiment for organizing related runs
mlflow.set_experiment(experiment_name)

In [ ]:
# Generate sample data
X, y = make_regression(n_samples=100, n_features=3, noise=0.1, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Create fit and predict with a model
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)
y_pred = lr_model.predict(X_test)

# Calculate some metrics
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

In [ ]:
# Experiment 1: Linear Regression
# Start a new MLflow run to track this experiment
with mlflow.start_run(run_name="Linear_Regression"):
    # Log model hyperparameters as key-value pairs
    mlflow.log_param("model_type", "LinearRegression")
    # Log evaluation metrics for comparison
    mlflow.log_metric("mse", mse)
    mlflow.log_metric("r2", r2)
    # Save the trained model to MLflow tracking server
    mlflow.sklearn.log_model(lr_model, name="model")

    print(f"Linear Regression - MSE: {mse:.4f}, R2: {r2:.4f}")

In [ ]:
# Experiment 2: Ridge Regression
# Start a new MLflow run to track this experiment
with mlflow.start_run(run_name="Ridge_Regression"):
    ridge_model = Ridge(alpha=1.0)
    ridge_model.fit(X_train, y_train)
    y_pred = ridge_model.predict(X_test)

    mse = mean_squared_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)

    # Log model hyperparameters as key-value pairs
    mlflow.log_param("model_type", "Ridge")
    mlflow.log_param("alpha", 1.0)
    # Log evaluation metrics for model comparison
    mlflow.log_metric("mse", mse)
    mlflow.log_metric("r2", r2)
    # Save the trained sklearn model (use mlflow.<lib>.log_model for other frameworks)
    mlflow.sklearn.log_model(ridge_model, name="model")

    print(f"Ridge Regression - MSE: {mse:.4f}, R2: {r2:.4f}")

# Terminal commands:
# Launch MLflow UI web interface on http://localhost:5000 to visualize runs
mlflow ui

# Kill any existing MLflow server if port is already in use
pkill -f mlflow

# Load and use a logged model

In [ ]:
import mlflow
import mlflow.sklearn

# Get the run ID from MLflow UI (experiment -> run -> Run ID on right panel)
run_id = "0c2520fed3914d7ea8eacb927dff5643"  # Replace with your actual run ID

# Build the model URI in format: runs:/<run_id>/<artifact_path>
model_uri = f"runs:/{run_id}/model"
# Load the logged sklearn model from the specified run
loaded_model = mlflow.sklearn.load_model(model_uri)

print("✅ Model loaded successfully!")
print(f"Model type: {type(loaded_model)}")

In [ ]:
loaded_model.predict(X_test)

In [ ]:
# Use the model for predictions
import numpy as np
sample_data = np.array([[1.0, 2.0, 3.0]])
prediction = loaded_model.predict(sample_data)
print(f"Prediction: {prediction}")

It means the model is forecasting ~232 total unit sales for a single day across all stores.

This specific value (231.59201983) comes from the MLflow notebook where a Linear Regression model is run on arbitrary dummy input 
[[1.0, 2.0, 3.0]] — so it's not a real prediction, just a demo output.